In [1]:
# 安装 PDF / Word 解析库
!pip install -q pdfplumber python-docx

导入库与全局配置

In [2]:
import os, re, json, time, textwrap
from openai import OpenAI
import pdfplumber
from docx import Document

# ===== 风格 / 岗位 / 轮数 映射 =====
STYLE_CONFIG = {
    "大厂标准": {
        "rounds": 5,
        "jobs": ["LLM应用算法工程师", "算法工程师", "机器学习工程师",
                 "AI工程师", "后端开发工程师", "数据工程师"]
    },
    "压力型": {
        "rounds": 5,
        "jobs": ["后端开发工程师", "算法工程师", "机器学习工程师",
                 "LLM应用算法工程师", "数据工程师", "系统开发工程师"]
    },
    "温和型": {
        "rounds": 3,
        "jobs": ["数据分析师", "测试工程师", "前端开发工程师",
                 "运维工程师", "产品经理（技术方向）", "初级开发工程师"]
    }
}

MODEL = "glm-5.2"   # 后续可换
MAX_RETRY = 3             # API / 校验重试次数

读取 Kaggle Secrets 并创建客户端

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ZHIPU_API_KEY = user_secrets.get_secret("ZHIPU_API_KEY")
client = OpenAI(
    api_key=ZHIPU_API_KEY,
    base_url="https://open.bigmodel.cn/api/paas/v4"
)
print("客户端创建成功")

客户端创建成功


通用 API 调用（指数退避重试）


In [4]:
def chat(messages, temperature=0.7, json_mode=False, max_retry=MAX_RETRY):
    """带指数退避重试的 API 调用"""
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    last_err = None
    for attempt in range(max_retry):
        try:
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            wait = 2 ** attempt
            print(f"[API重试 {attempt+1}/{max_retry}] {e}，{wait}s 后重试")
            time.sleep(wait)
    raise RuntimeError(f"API 调用失败：{last_err}")

简历解析（PDF / Word → 文本）

In [5]:
def parse_resume(file_path):
    """根据后缀解析 PDF / Word 为纯文本"""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        text = []
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text.append(page.extract_text() or "")
        return "\n".join(text).strip()

    elif ext in (".docx", ".doc"):
        doc = Document(file_path)
        return "\n".join(p.text for p in doc.paragraphs).strip()

    else:
        raise ValueError("仅支持 PDF / Word(.docx) 文件")

解析简历 + 提取岗位 + 自动匹配风格

In [6]:
# ===== 用户在此填写文件路径 =====
RESUME_PATH = "/kaggle/input/datasets/ysqsww/resume/C.docx"   # 改成你的路径

resume_text = parse_resume(RESUME_PATH)
print("===== 简历解析结果（前500字）=====")
print(resume_text[:500])

===== 简历解析结果（前500字）=====
赵六
电话：137-0000-0000 | 邮箱：zhaoliu@example.com
求职意向：数据分析师

教育背景
某大学  软件工程专业  本科  2021.09–2025.06

项目经历
学生成绩管理系统（课程作业）  2023.09–2023.12
- 角色：课程项目成员
- 技术栈：Python, Java
- 职责：完成基础功能开发。

实习经历
无

技能
Python, Java


In [7]:
def extract_job(resume_text):
    """让 AI 从简历中提取求职意向岗位"""
    prompt = f"""请从下面简历中提取“求职意向”的岗位名称，
只输出岗位名，不要任何解释。

简历：
{resume_text[:1500]}
"""
    return chat([{"role": "user", "content": prompt}], temperature=0).strip()

job = extract_job(resume_text)
print("提取到的求职意向：", job)

提取到的求职意向： 数据分析师


In [8]:
def match_style(job):
    """根据岗位匹配风格，匹配不到返回 None"""
    for style, cfg in STYLE_CONFIG.items():
        for j in cfg["jobs"]:
            if j in job or job in j:
                return style
    return None

style = match_style(job)

if style is None:
    print("未匹配到岗位组，请手动选择风格：")
    for i, s in enumerate(STYLE_CONFIG, 1):
        print(f"{i}. {s}")
    choice = input("请输入序号：").strip()
    style = list(STYLE_CONFIG.keys())[int(choice) - 1]

rounds = STYLE_CONFIG[style]["rounds"]
print(f"最终风格：{style}，轮数：{rounds}")

最终风格：温和型，轮数：3


三类 System Prompt + 三个独立 messages

In [9]:
INTERVIEWER_PROMPT = """你是一名{style}风格的技术面试官，岗位是{job}。
规则：
1. 只能基于候选人简历和已有对话提问，禁止编造简历中不存在的经历。
2. 每轮只问一个问题。
3. 从第2轮起，必须引用候选人上一轮回答中的具体内容（关键词/说法）继续追问。
4. 如果候选人回答模糊（如“良好效果”），必须追问可验证的细节。
5. 如果简历经验空白，转向课程基础、技能或求职意愿，不要编造项目。
6. 若被要求“引用XX继续追问”，必须照做。
"""

CANDIDATE_PROMPT = """你扮演候选人，仅用于占位，实际回答由用户输入。"""

JUDGE_PROMPT = """你是一名严格的面试评分官。请根据完整面试对话输出 JSON，字段：
{
  "总分": 0-100,
  "技术深度": 0-25,
  "逻辑表达": 0-25,
  "项目真实性": 0-25,
  "沟通能力": 0-25,
  "优势": "...",
  "不足": "...",
  "改进建议": "...",
  "复习清单": ["...", "..."],
  "一句话总评": "..."
}
要求：评分必须有区分度，四个维度极差至少5分，禁止全部接近满分。
只输出 JSON，不要多余文字。
"""

# 三个独立上下文
interviewer_msgs = [{"role": "system",
                     "content": INTERVIEWER_PROMPT.format(style=style, job=job)}]
candidate_msgs   = [{"role": "system", "content": CANDIDATE_PROMPT}]
judge_msgs       = [{"role": "system", "content": JUDGE_PROMPT}]

# 完整对话记录（供评分官使用）
full_dialogue = [f"岗位：{job}", f"风格：{style}", f"简历：\n{resume_text}"]

校验工具函数（追问引用 / 编造检测）

In [10]:
STOPWORDS = set("的 了 和 是 在 我 有 就 不 人 都 一 一个 上 也 很 到 说 要 去 你 会 着 没有 看 好 自己 这 那 与 及 等 中 对 为 以 并 而 但 如果 因为 所以".split())

def extract_keywords(text, topn=8):
    """粗略提取实词关键词"""
    words = re.findall(r"[A-Za-z0-9\u4e00-\u9fa5]+", text)
    words = [w for w in words if w not in STOPWORDS and len(w) > 1]
    # 按出现顺序去重
    seen, result = set(), []
    for w in words:
        if w not in seen:
            seen.add(w)
            result.append(w)
    return result[:topn]

def question_references(question, prev_answer):
    """检查问题是否引用了上一轮回答的关键词"""
    kws = extract_keywords(prev_answer)
    if not kws:
        return True  # 无关键词则跳过校验
    return any(kw in question for kw in kws)

def detect_fabrication(question, resume_text):
    """检测问题是否出现简历中不存在的项目/技术词（粗略版）"""
    tech_terms = re.findall(r"[A-Za-z][A-Za-z0-9\-\.]{2,}", question)
    fabricated = []
    for t in set(tech_terms):
        if t.lower() not in resume_text.lower():
            fabricated.append(t)
    return fabricated

面试主循环

In [11]:
def generate_question(round_idx, prev_answer=None, force_kw=None):
    """生成一题，含引用校验与编造校验"""
    # 把完整对话喂给面试官（隔离上下文：只放面试官自己的历史问答）
    prompt = f"现在是第 {round_idx} 轮，请提问。"
    if prev_answer:
        prompt += f"\n候选人上一轮回答：{prev_answer}"
    if force_kw:
        prompt += f"\n请引用“{force_kw}”继续追问。"

    interviewer_msgs.append({"role": "user", "content": prompt})

    for attempt in range(MAX_RETRY):
        q = chat(interviewer_msgs, temperature=0.7).strip()

        # 校验1：第2轮起必须引用上一轮关键词
        if round_idx >= 2 and prev_answer and not question_references(q, prev_answer):
            interviewer_msgs.append({"role": "assistant", "content": q})
            interviewer_msgs.append({"role": "user",
                "content": "你的问题没有引用候选人上一轮回答的具体内容，请重新提问。"})
            continue

        # 校验2：编造检测
        fab = detect_fabrication(q, resume_text)
        if fab:
            interviewer_msgs.append({"role": "assistant", "content": q})
            interviewer_msgs.append({"role": "user",
                "content": f"问题中出现了简历里不存在的词 {fab}，请勿编造经历，重新提问。"})
            continue

        interviewer_msgs.append({"role": "assistant", "content": q})
        return q

    # 兜底：强制引用关键词
    kw = extract_keywords(prev_answer, topn=1)
    kw = kw[0] if kw else "你刚才提到的内容"
    interviewer_msgs.append({"role": "user",
        "content": f"请引用“{kw}”继续追问，只问一个问题。"})
    q = chat(interviewer_msgs, temperature=0.7).strip()
    interviewer_msgs.append({"role": "assistant", "content": q})
    return q


print("===== 面试开始 =====")
prev_answer = None

for r in range(1, rounds + 1):
    q = generate_question(r, prev_answer)
    print(f"\n【第{r}轮】面试官：{q}")

    ans = input("你的回答：").strip()
    prev_answer = ans

    full_dialogue.append(f"面试官（第{r}轮）：{q}")
    full_dialogue.append(f"候选人（第{r}轮）：{ans}")

print("\n===== 面试结束 =====")

===== 面试开始 =====

【第1轮】面试官：你好，欢迎参加今天的数据分析师面试，不用紧张，我们就当是一次轻松的技术交流。

为了能更有针对性地和你探讨，你能先分享一下你的简历，或者简单介绍一下你过往经历中最让你有成就感的一个数据分析项目吗？


你的回答： 面试官您好，我叫赵六，本科是软件工程专业。我之前没有正式的数据分析实习，在校期间主要做开发类课程项目，同时自学数据分析相关能力，比较擅长用 Python 做数据处理。  我最有感触的项目是课程里的学生成绩管理系统。当时我负责后端基础功能开发，在做的过程里，我不只是完成增删改查，还额外基于系统里的成绩数据做了简单的数据分析：用 Python 读取成绩表，计算各科平均分、不及格率，按班级和考试批次做分组统计，找出明显拉低整体通过率的科目，生成统计报表给任课老师参考。  在这个小分析里，我用 Pandas 清洗缺失、异常的成绩数据，用 Matplotlib 画出分数分布直方图，识别偏科严重的群体。最后输出的统计结果帮助老师定位了教学难点。  通过这件事我发现自己更愿意从数据里发现业务问题，所以打算转向数据分析师方向。我目前掌握 Python、Java，熟悉 Pandas 做数据清洗、指标统计，也在持续学习 SQL、可视化和基础数据分析思路。



【第2轮】面试官：赵六你好，你开头说"面试官您好"，然后提到自己"比较擅长用 Python 做数据处理"。我想具体了解一下，你在那个学生成绩项目里用 Pandas 做数据清洗时，具体调用了哪些方法来处理缺失值和异常值的？能举一个实际处理的例子吗？


你的回答： 在这个成绩项目里，我先用`df.isnull()`查看缺失分布，再根据业务含义选择处理方式；异常值主要通过描述统计和范围判断来识别。  先说缺失值：成绩表里有部分缺考的记录，分数字段是空值。缺考本身是业务含义，不能随便填平均分，所以我用`df['score'].fillna(0)`把缺考分数填充为 0，同时新增标记列区分是正常低分还是缺考；而如果是少量录入失误、没有业务含义的缺失，我会用`dropna()`直接删掉该行，避免干扰平均分计算。  异常值这块，分数合理区间是 0-100，我用`df[(df['score']<0)|(df['score']>100)]`筛选，找到了几条录入错误的 120、-5 这类脏数据。当时我没有直接删除，先核对原始记录，确认是手误录入，最后用`df.loc`修改为正确分值。  举个实际例子：有一条数据 score 是 120，明显超出满分上限，我先通过`describe()`看整体分数分布发现这个极值，然后定位到这条记录，和系统原始表单对比，确认是老师录入时多打了 20，把它改成 100。如果是无法溯源、确认不了的异常分数，我会直接剔除，防止拉高或拉低班级平均分、不及格率这些指标。  用到的 Pandas 方法主要是`isnull()`、`fillna()`、`dropna()`、布尔索引筛选、`describe()`。



【第3轮】面试官：你刚才提到"在这个成绩项目里"用描述统计和范围判断来识别异常值，发现了一条 score 为 120 的记录。我想追问一下：除了这种超出 0-100 范围的明显异常，你当时有没有遇到过数据本身在合理范围内但仍然可能影响分析结果的情况？比如某个班级某次考试整体成绩异常偏低，你是怎么判断这是真实情况还是数据本身有问题的？


你的回答： 有的，项目里确实遇到过分数落在 0-100 区间、单看数值合法，但整体分布反常的情况。遇到某个班级单次考试平均分明显低于其他班的时候，我会分几步交叉验证，区分是真实情况还是数据问题。  首先我会做**分组对比**：用 Pandas 按班级、考试批次分组，算出各班的平均分、中位数、标准差，同时画出每个班的分数直方图。如果这个低分班级中位数也很低、大部分分数集中在 20-40 分，而同期别的班级集中在 60-80 分，先标记为异常群体，不直接判定数据脏了。  然后做数据层面的排查，优先找数据 bug： 第一，核对筛选条件，看是不是不小心把 “缺考标记 = 0” 的记录全都算进来了，比如缺考填充的 0 分大量混入这个班级，拉低均值； 第二，检查数据来源，确认是不是导入时串表了，比如把别的科目成绩错误导入到本次考试； 第三，检查样本量，是不是这个班只导入了一小部分学生，样本不完整，导致指标失真。  如果上面这些数据问题都排除了，我再去做业务核验，也就是找任课老师确认：比如这次试卷难度偏高、班级整体复习不到位，或是考试当天有特殊情况，那就是真实的成绩现象，分析报告里要保留这批数据，并且在结论里备注背景原因，不能直接删除。  举个当时的例子：有一个班平均分只有 42 分，所有分数都在 0-100 之间。我排查后发现，这个班有十几个人是缺考，我之前填充的 0 分没有单独过滤，直接参与平均分计算，拉低了整体指标。后面我把缺考样本单独剥离，再重新计算有效考生的平均分，修正到 61 分，就和整体水平匹配了。  总结来说，我不会只依靠单条记录的数值范围判断，而是**先校验数据链路、样本口径，再结合业务背景确认**；数据问题就清洗修正，真实的业务现象就保留并标注，避免误删有效数据。



===== 面试结束 =====


评分官生成报告（JSON 校验 + 清洗重试）

In [12]:
def clean_json(text):
    """清洗 ```json 包裹等"""
    text = text.strip()
    text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()
    return text

def judge_score(dialogue_text):
    judge_msgs.append({"role": "user",
        "content": "以下是完整面试对话，请输出评分 JSON：\n" + dialogue_text})

    for attempt in range(MAX_RETRY):
        raw = chat(judge_msgs, temperature=0.3, json_mode=True)
        try:
            data = json.loads(clean_json(raw))
        except Exception as e:
            print(f"[JSON解析失败 {attempt+1}] {e}，重试")
            judge_msgs.append({"role": "assistant", "content": raw})
            judge_msgs.append({"role": "user", "content": "上次输出不是合法JSON，请只输出合法JSON。"})
            continue

        # 区分度校验：四维度极差 >= 5
        dims = [data.get("技术深度",0), data.get("逻辑表达",0),
                data.get("项目真实性",0), data.get("沟通能力",0)]
        if max(dims) - min(dims) < 5:
            print(f"[区分度不足 {attempt+1}] 极差={max(dims)-min(dims)}，重评")
            judge_msgs.append({"role": "assistant", "content": raw})
            judge_msgs.append({"role": "user",
                "content": "四个维度分数过于集中，请拉开差距，极差至少5分，重新输出JSON。"})
            continue

        return data, raw

    raise RuntimeError("评分官多次未能输出合规 JSON")

report, raw_json = judge_score("\n".join(full_dialogue))

格式化展示报告 + 保留原始 JSON

In [13]:
def print_report(data):
    print("=" * 40)
    print(f"总分：{data.get('总分')} / 100")
    print("-" * 40)
    print(f"技术深度：{data.get('技术深度')} / 25")
    print(f"逻辑表达：{data.get('逻辑表达')} / 25")
    print(f"项目真实性：{data.get('项目真实性')} / 25")
    print(f"沟通能力：{data.get('沟通能力')} / 25")
    print("-" * 40)
    print("优势：", data.get("优势"))
    print("不足：", data.get("不足"))
    print("改进建议：", data.get("改进建议"))
    print("复习清单：")
    for item in data.get("复习清单", []):
        print("  -", item)
    print("一句话总评：", data.get("一句话总评"))
    print("=" * 40)

print_report(report)

print("\n===== 原始 JSON（备用）=====")
print(raw_json)

总分：77 / 100
----------------------------------------
技术深度：18 / 25
逻辑表达：22 / 25
项目真实性：16 / 25
沟通能力：21 / 25
----------------------------------------
优势： 逻辑思维严密，对数据异常值的排查思路非常清晰，能结合业务场景（如缺考与低分区分、样本量校验）进行数据清洗，具备优秀的数据敏感度和初级分析师潜质；表达结构化强，沟通顺畅。
不足： 缺乏真实业务实习经历，项目仅停留在课程作业层面；技术栈相对单一（主要依赖Python/Pandas），缺乏SQL、BI工具及更高级统计学方法的实战经验证明。
改进建议： 建议补充SQL相关实战项目，熟悉多表关联、窗口函数等分析师常用技能；尝试寻找一段正式的实习，将分析思路应用到更复杂的商业场景中；学习基础的A/B测试或统计学假设检验方法。
复习清单：
  - SQL基础与进阶（多表Join、窗口函数）
  - 统计学基础（假设检验、A/B测试）
  - BI可视化工具（如Tableau或PowerBI）
  - 业务分析框架（如漏斗分析、留存分析等）
一句话总评： 作为无实习经验的应届生，展现了出色的数据敏感度和严谨的逻辑排查能力，基础扎实，具备培养潜力，但需补充SQL等核心技能与真实业务经验。

===== 原始 JSON（备用）=====
{"总分":77,"技术深度":18,"逻辑表达":22,"项目真实性":16,"沟通能力":21,"优势":"逻辑思维严密，对数据异常值的排查思路非常清晰，能结合业务场景（如缺考与低分区分、样本量校验）进行数据清洗，具备优秀的数据敏感度和初级分析师潜质；表达结构化强，沟通顺畅。","不足":"缺乏真实业务实习经历，项目仅停留在课程作业层面；技术栈相对单一（主要依赖Python/Pandas），缺乏SQL、BI工具及更高级统计学方法的实战经验证明。","改进建议":"建议补充SQL相关实战项目，熟悉多表关联、窗口函数等分析师常用技能；尝试寻找一段正式的实习，将分析思路应用到更复杂的商业场景中；学习基础的A/B测试或统计学假设检验方法。","复习清单":["SQL基础与进阶（多表Join、窗口函数）","统计学基础（假设检验、A/B测试）","BI可视化工具